In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Student Academic Performance Prediction App\n",
    "\n",
    "This notebook develops a student academic performance prediction application using machine learning. It includes dataset generation, exploration, preprocessing, model training, evaluation, and a simple Streamlit web app prototype.\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Import Libraries\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import matplotlib\n",
    "matplotlib.use('Agg')\n",
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import seaborn as sns\n",
    "from sklearn.ensemble import RandomForestRegressor\n",
    "from sklearn.linear_model import LinearRegression\n",
    "from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.pipeline import Pipeline\n",
    "from sklearn.preprocessing import StandardScaler\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Load or Create Student Performance Dataset\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def generate_student_dataset(num_records=500):\n",
    "    np.random.seed(42)\n",
    "    study_hours = np.round(np.random.uniform(0.5, 8.0, size=num_records), 1)\n",
    "    attendance = np.round(np.random.uniform(50.0, 100.0, size=num_records), 1)\n",
    "    assignments = np.random.randint(0, 11, size=num_records)\n",
    "    previous_marks = np.round(np.random.uniform(25.0, 100.0, size=num_records), 1)\n",
    "    participation = np.random.randint(1, 6, size=num_records)\n",
    "\n",
    "    raw_score = (\n",
    "        study_hours * 6.0\n",
    "        + attendance * 0.25\n",
    "        + assignments * 3.5\n",
    "        + previous_marks * 0.35\n",
    "        + participation * 2.0\n",
    "    )\n",
    "    noise = np.random.normal(0, 6, size=num_records)\n",
    "    final_grade = np.clip(np.round(raw_score + noise, 1), 20.0, 100.0)\n",
    "\n",
    "    df = pd.DataFrame({\n",
    "        'study_hours_per_day': study_hours,\n",
    "        'attendance_percentage': attendance,\n",
    "        'assignments_completed': assignments,\n",
    "        'previous_semester_marks': previous_marks,\n",
    "        'class_participation': participation,\n",
    "        'final_grade': final_grade,\n",
    "    })\n",
    "    return df\n",
    "\n",
    "student_df = generate_student_dataset(500)\n",
    "student_df.head()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Explore and Visualize Data\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "student_df.describe()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(12, 8))\n",
    "sns.histplot(student_df['final_grade'], bins=20, kde=True, color='skyblue')\n",
    "plt.title('Final Grade Distribution')\n",
    "plt.xlabel('Final Grade')\n",
    "plt.ylabel('Count')\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(12, 8))\n",
    "sns.heatmap(student_df.corr(), annot=True, cmap='Blues', fmt='.2f')\n",
    "plt.title('Feature Correlation Matrix')\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Preprocess Data\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "student_df = student_df.drop_duplicates()\n",
    "student_df = student_df.fillna(student_df.mean(numeric_only=True))\n",
    "student_df.isna().sum()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "features = [\n",
    "    'study_hours_per_day',\n",
    "    'attendance_percentage',\n",
    "    'assignments_completed',\n",
    "    'previous_semester_marks',\n",
    "    'class_participation',\n",
    "]\n",
    "X = student_df[features]\n",
    "y = student_df['final_grade']\n",
    "\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42\n",
    ")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Train Machine Learning Model\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "models = {\n",
    "    'Linear Regression': Pipeline([\n",
    "        ('scaler', StandardScaler()),\n",
    "        ('regressor', LinearRegression())\n",
    "    ]),\n",
    "    'Random Forest Regressor': RandomForestRegressor(n_estimators=300, random_state=42)\n",
    "}\n",
    "\n",
    "results = {}\n",
    "for name, model in models.items():\n",
    "    model.fit(X_train, y_train)\n",
    "    y_pred = model.predict(X_test)\n",
    "    mae = mean_absolute_error(y_test, y_pred)\n",
    "    rmse = mean_squared_error(y_test, y_pred, squared=False)\n",
    "    r2 = r2_score(y_test, y_pred)\n",
    "    results[name] = {'mae': mae, 'rmse': rmse, 'r2': r2}\n",
    "    print(f'{name}: MAE={mae:.2f}, RMSE={rmse:.2f}, R2={r2:.4f}')\n",
    "\n",
    "best_model_name = max(results, key=lambda k: results[k]['r2'])\n",
    "print(f'Best model: {best_model_name}')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Evaluate Model Performance\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(10, 6))\n",
    "plt.scatter(y_test, model.predict(X_test), alpha=0.7, color='navy')\n",
    "plt.plot([20, 100], [20, 100], color='darkorange', linestyle='--')\n",
    "plt.xlabel('Actual Grade')\n",
    "plt.ylabel('Predicted Grade')\n",
    "plt.title('Actual vs. Predicted Final Grades')\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Save Model and Prepare for Web Integration\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pickle\n",
    "\n",
    "best_model = models[best_model_name]\n",
    "best_model.fit(X_train, y_train)\n",
    "\n",
    "with open('student_performance_model.pkl', 'wb') as f:\n",
    "    pickle.dump(best_model, f)\n",
    "\n",
    "print('Saved best model to student_performance_model.pkl')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Build Simple Streamlit App Prototype\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "streamlit_code = '''\n",
    "import pickle\n",
    "import streamlit as st\n",
    "import numpy as np\n",
    "\n",
    "with open('student_performance_model.pkl', 'rb') as model_file:\n",
    "    model = pickle.load(model_file)\n",
    "\n",
    "st.title('Student Academic Performance Predictor')\n",
    "st.write('Enter student details to predict the final grade.')\n",
    "\n",
    "study_hours = st.number_input('Study Hours Per Day', min_value=0.0, max_value=12.0, value=2.0, step=0.1)\n",
    "attendance = st.number_input('Attendance Percentage', min_value=0.0, max_value=100.0, value=80.0, step=0.1)\n",
    "assignments = st.number_input('Assignments Completed', min_value=0, max_value=10, value=6)\n",
    "previous_marks = st.number_input('Previous Semester Marks', min_value=0.0, max_value=100.0, value=70.0, step=0.1)\n",
    "participation = st.number_input('Class Participation (1-5)', min_value=1, max_value=5, value=3)\n",
    "\n",
    "if st.button('Predict Performance'):\n",
    "    features = np.array([[study_hours, attendance, assignments, previous_marks, participation]])\n",
    "    prediction = model.predict(features)[0]\n",
    "    st.success(f'Predicted Final Grade: {prediction:.1f}')\n",
    "'''
    "\n",
    "with open('streamlit_app.py', 'w') as f:\n",
    "    f.write(streamlit_code)\n",
    "\n",
    "print('Created Streamlit app prototype in streamlit_app.py')\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.13.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Student Academic Performance Prediction App\n",
    "\n",
    "This notebook develops a student academic performance prediction application using machine learning. It includes dataset generation, exploration, preprocessing, model training, evaluation, and a simple Streamlit web app prototype.\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Import Libraries\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import pickle\n",
    "\n",
    "import matplotlib\n",
    "matplotlib.use('Agg')\n",
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import seaborn as sns\n",
    "from sklearn.ensemble import RandomForestRegressor\n",
    "from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.pipeline import Pipeline\n",
    "from sklearn.preprocessing import MinMaxScaler\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Load or Create Student Performance Dataset\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def generate_student_dataset(num_records=500):\n",
    "    np.random.seed(42)\n",
    "    study_hours = np.round(np.random.uniform(0.5, 8.0, size=num_records), 1)\n",
    "    attendance = np.round(np.random.uniform(50.0, 100.0, size=num_records), 1)\n",
    "    assignments = np.random.randint(0, 11, size=num_records)\n",
    "    previous_marks = np.round(np.random.uniform(25.0, 100.0, size=num_records), 1)\n",
    "    participation = np.random.randint(1, 6, size=num_records)\n",
    "\n",
    "    raw_score = (\n",
    "        study_hours * 6.0\n",
    "        + attendance * 0.25\n",
    "        + assignments * 3.5\n",
    "        + previous_marks * 0.35\n",
    "        + participation * 2.0\n",
    "    )\n",
    "    noise = np.random.normal(0, 6, size=num_records)\n",
    "    final_grade = np.clip(np.round(raw_score + noise, 1), 20.0, 100.0)\n",
    "\n",
    "    df = pd.DataFrame({\n",
    "        'study_hours_per_day': study_hours,\n",
    "        'attendance_percentage': attendance,\n",
    "        'assignments_completed': assignments,\n",
    "        'previous_semester_marks': previous_marks,\n",
    "        'class_participation': participation,\n",
    "        'final_grade': final_grade,\n",
    "    })\n",
    "    return df\n",
    "\n",
    "student_df = generate_student_dataset(500)\n",
    "student_df.head()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Explore and Visualize Data\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "student_df.describe()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(12, 8))\n",
    "sns.histplot(student_df['final_grade'], bins=20, kde=True, color='skyblue')\n",
    "plt.title('Final Grade Distribution')\n",
    "plt.xlabel('Final Grade')\n",
    "plt.ylabel('Count')\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(12, 8))\n",
    "sns.heatmap(student_df.corr(), annot=True, cmap='Blues', fmt='.2f')\n",
    "plt.title('Feature Correlation Matrix')\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Preprocess Data\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "student_df.isna().sum()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "features = [\n",
    "    'study_hours_per_day',\n",
    "    'attendance_percentage',\n",
    "    'assignments_completed',\n",
    "    'previous_semester_marks',\n",
    "    'class_participation',\n",
    "]\n",
    "X = student_df[features]\n",
    "y = student_df['final_grade']\n",
    "\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42\n",
    ")\n",
    "\n",
    "scaler = MinMaxScaler()\n",
    "X_train_scaled = scaler.fit_transform(X_train)\n",
    "X_test_scaled = scaler.transform(X_test)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Train Machine Learning Model\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "model = RandomForestRegressor(n_estimators=100, random_state=42)\n",
    "model.fit(X_train_scaled, y_train)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Evaluate Model Performance\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "y_pred = model.predict(X_test_scaled)\n",
    "mae = mean_absolute_error(y_test, y_pred)\n",
    "rmse = np.sqrt(mean_squared_error(y_test, y_pred))\n",
    "r2 = r2_score(y_test, y_pred)\n",
    "\n",
    "print(f'Mean Absolute Error: {mae:.2f}')\n",
    "print(f'RMSE: {rmse:.2f}')\n",
    "print(f'R2 Score: {r2:.2f}')\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(10, 6))\n",
    "plt.scatter(y_test, y_pred, alpha=0.7, color='navy')\n",
    "plt.plot([20, 100], [20, 100], color='darkorange', linestyle='--')\n",
    "plt.xlabel('Actual Grade')\n",
    "plt.ylabel('Predicted Grade')\n",
    "plt.title('Actual vs. Predicted Final Grades')\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Save Model and Prepare for Web Integration\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "model_artifact_path = 'student_performance_model.pkl'\n",
    "preprocessor_artifact_path = 'scaler.pkl'\n",
    "\n",
    "with open(model_artifact_path, 'wb') as f:\n",
    "    pickle.dump(model, f)\n",
    "\n",
    "with open(preprocessor_artifact_path, 'wb') as f:\n",
    "    pickle.dump(scaler, f)\n",
    "\n",
    "print(f'Saved model to {model_artifact_path}')\n",
    "print(f'Saved scaler to {preprocessor_artifact_path}')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Build Simple Streamlit App Prototype\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "streamlit_code = '''\n",
    "import pickle\n",
    "import streamlit as st\n",
    "import numpy as np\n",
    "\n",
    "with open('student_performance_model.pkl', 'rb') as model_file:\n",
    "    model = pickle.load(model_file)\n",
    "\n",
    "with open('scaler.pkl', 'rb') as scaler_file:\n",
    "    scaler = pickle.load(scaler_file)\n",
    "\n",
    "st.title('Student Academic Performance Predictor')\n",
    "st.write('Enter student details to predict the final grade.')\n",
    "\n",
    "study_hours = st.number_input('Study Hours Per Day', min_value=0.0, max_value=12.0, value=2.0, step=0.1)\n",
    "attendance = st.number_input('Attendance Percentage', min_value=0.0, max_value=100.0, value=80.0, step=0.1)\n",
    "assignments = st.number_input('Assignments Completed', min_value=0, max_value=10, value=6)\n",
    "previous_marks = st.number_input('Previous Semester Marks', min_value=0.0, max_value=100.0, value=70.0, step=0.1)\n",
    "participation = st.number_input('Class Participation (1-5)', min_value=1, max_value=5, value=3)\n",
    "\n",
    "if st.button('Predict Performance'):\n",
    "    features = np.array([[study_hours, attendance, assignments, previous_marks, participation]])\n",
    "    features_scaled = scaler.transform(features)\n",
    "    prediction = model.predict(features_scaled)[0]\n",
    "    st.success(f'Predicted Final Grade: {prediction:.1f}')\n",
    "    if prediction >= 85:\n",
    "        st.info('Performance category: Excellent')\n",
    "    elif prediction >= 75:\n",
    "        st.info('Performance category: Good')\n",
    "    elif prediction >= 60:\n",
    "        st.info('Performance category: Average')\n",
    "    else:\n",
    "        st.info('Performance category: Needs Improvement')\n",
    "'''\n",
    "\n",
    "with open('streamlit_app.py', 'w') as f:\n",
    "    f.write(streamlit_code)\n",
    "\n",
    "print('Created Streamlit app prototype in streamlit_app.py')\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.13.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


: 